# Du2015 — Split-window

Estima a temperatura da superfície (LST) combinando as temperaturas de brilho B10/B11 e a emissividade de cada banda. Pode usar coeficientes gerais ou selecionados pela quantidade de vapor d’água informada.

## Precisa de

**Temperaturas de brilho — obrigatórias nas duas formas de uso:**

- **BT10:** imagem de temperatura de brilho TOA da banda 10, em **K (kelvin)**.
- **BT11:** imagem de temperatura de brilho TOA da banda 11, em **K (kelvin)**.

Essas entradas são temperaturas de brilho, ainda sem a correção de emissividade para obter LST. Não fornecer DN, radiância ou temperaturas em Celsius nesses dois parâmetros.

**Emissividade — escolha uma das duas formas abaixo.**

**1. Calcular dentro da função, como neste notebook:**

- **B4 — vermelho:** imagem calibrada, em **W m⁻² sr⁻¹ µm⁻¹** (radiância TOA, nesta configuração).
- **B5 — infravermelho próximo (NIR):** imagem calibrada, em **W m⁻² sr⁻¹ µm⁻¹**, na mesma convenção da B4.

A função calcula o NDVI internamente e estima as duas emissividades com `emissivity_method="gopinadh-2018"`. O NDVI é adimensional e não precisa ser fornecido separadamente. A escolha de radiância neste exemplo acompanha a convenção declarada para essa parametrização; trocar a grandeza exige verificar a compatibilidade do modelo.

**2. Fornecer emissividades já calculadas:**

- **`emissivity_10`:** imagem de emissividade da banda 10, **adimensional**, com valores em **(0,1]**.
- **`emissivity_11`:** imagem de emissividade da banda 11, **adimensional**, com valores em **(0,1]**.

Nessa forma, fornecer as duas imagens e omitir B4, B5 e `emissivity_method` na chamada. Cada emissividade deve corresponder à sua própria banda.

**Entradas opcionais:**

- **`water_vapor`:** vapor d’água integrado na coluna atmosférica (PWV), em **g/cm²**, como número único ou imagem alinhada. Du aceita **0 a 6,3 g/cm²**. Com `None`, usa os coeficientes gerais; não estima PWV automaticamente.
- **`mask`:** imagem **booleana**, sem unidade: `True` exclui o pixel; `False` permite avaliar. Neste notebook, o leitor monta a máscara a partir dos arquivos de qualidade, conforme explicado na seção de leitura.

Todas as imagens usadas juntas devem ter a mesma quantidade de linhas/colunas, posição dos pixels, resolução e sistema de coordenadas.

## Arquivos para executar este notebook

O exemplo parte de uma cena **Landsat 8/9 Collection 2 Level 1**. Separe na pasta da cena:

- **B4:** arquivo original da banda vermelha, em **DN**; o leitor calibra para radiância.
- **B5:** arquivo original da banda NIR, em **DN**; o leitor calibra para radiância.
- **B10:** arquivo térmico original, em **DN**; o leitor converte para BT10 em **K**.
- **B11:** arquivo térmico original, em **DN**; o leitor converte para BT11 em **K**.
- **MTL.txt:** metadados e coeficientes de calibração da própria cena; não é uma imagem.
- **QA_PIXEL:** códigos de qualidade por pixel; o leitor interpreta seus bits para montar a máscara.
- **QA_RADSAT:** códigos de saturação e oclusão de terreno; o leitor interpreta seus bits para complementar a máscara.

Os arquivos QA contêm códigos inteiros, sem unidade física; não devem ser passados diretamente como máscara booleana. B2 e SZA são arquivos opcionais para o leitor. Preserve os nomes originais e a grade dos arquivos da cena.

Altere os caminhos na célula de configuração. O leitor realiza a calibração com o MTL; não é necessário preparar manualmente BT10/BT11 e as radiâncias para executar este exemplo. A chamada com emissividades prontas é uma alternativa da função; a leitura deste notebook continua usando os arquivos da cena listados acima.

## Como as imagens são preparadas

**TOA significa topo da atmosfera.** A grandeza também precisa ser especificada:

- **Radiância TOA:** energia medida pelo sensor, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Reflectância TOA:** **adimensional**, com ajuste do ângulo solar.
- **Temperatura de brilho TOA:** temperatura equivalente da radiância térmica, em **K**.

O leitor usa o **MTL da própria cena** para calibrar os arquivos originais. Não é necessário preparar manualmente as imagens TOA para executar este notebook.

A configuração `optical_quantity="radiance"` devolve as bandas ópticas em **radiância TOA, em W m⁻² sr⁻¹ µm⁻¹**; as bandas térmicas são devolvidas como temperaturas de brilho em **K**.

Trocar para `optical_quantity="reflectance"` muda a calibração óptica; isso exige verificar a compatibilidade com a parametrização usada no método. **Reflectância TOA ainda contém efeitos atmosféricos e é diferente de reflectância de superfície.**


## Função geral — parâmetros aplicáveis ao método

Argumentos nomeados e relacionados ficam próximos. A tabela abaixo segue **exatamente a ordem desta chamada**. Obrigatório: necessário ao cálculo; opcional: pode usar o padrão; condicional: depende da rota escolhida. Valores de configuração mostrados podem diferir do padrão da API.

```python
resultado = split_window(
    lst_method="du-2015",

    # Cálculo da emissividade — requer vermelho/NIR
    emissivity_method="gopinadh-2018",
    band_4_red=red,
    band_5_nir=nir,
    optical_quantity="radiance",

    # Temperaturas de brilho — K
    brightness_band_10=bt10,
    brightness_band_11=bt11,

    # Vapor d'água — g/cm²
    water_vapor=None,

    # Exclusão de pixels
    mask=mascara,

    # Saída
    unit="celsius",
    return_diagnostics=False,
)
```

## Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções para Du |
|---|---|---|---|---|
| `lst_method` | Obrigatório: selecionar o método | `None` (precisa informar) | — (seletor) | Usar `"du-2015"` neste notebook. |
| `emissivity_method` | Opcional na entrada por bandas; omitir com emissividades prontas | `None` → `"gopinadh-2018"` na entrada por bandas | — (seletor) | A seleção automática atual de split-window usa `"gopinadh-2018"`, com valores distintos para B10/B11. `"avdan-2016"` é bloqueado nesse fluxo. Para outro modelo de emissividade, calcule o par separadamente e forneça `emissivity_10`/`emissivity_11`. |
| `band_4_red` | Obrigatório neste exemplo; dispensável com emissividades prontas | `None` | Neste exemplo: W m⁻² sr⁻¹ µm⁻¹; reflectância é adimensional | Banda vermelha calibrada. Deve ter a mesma convenção radiométrica do NIR e ser compatível com a parametrização da emissividade. O NDVI é calculado internamente. Com emissividades prontas, deixar `None`. |
| `band_5_nir` | Obrigatório neste exemplo; dispensável com emissividades prontas | `None` | Mesma grandeza e unidade do vermelho | Banda NIR calibrada e alinhada. Não combinar DN de uma banda com radiância/reflectância da outra. Com emissividades prontas, deixar `None`. |
| `optical_quantity` | Opcional; recomendado declarar | `None` (não declarada) | — (declaração de grandeza) | `"radiance"`, `"reflectance"`/`"toa_reflectance"` ou `"surface_reflectance"`. Apenas registra a convenção dos arrays; **não converte** radiância em reflectância nem faz correção atmosférica. O leitor deste exemplo oferece `"radiance"` ou `"reflectance"` (TOA). |
| `brightness_band_10` | Obrigatório | Sem padrão | K (kelvin) | Raster de temperatura de brilho **TOA B10**, calibrado pelo MTL. Não fornecer DN, radiância, LST ou Celsius. |
| `brightness_band_11` | Obrigatório | Sem padrão | K (kelvin) | Raster de temperatura de brilho **TOA B11**, com a mesma forma e grade de B10. |
| `water_vapor` | Opcional | `None` | g/cm² | PWV escalar ou raster alinhado, entre **0 e 6,3**, inclusive. `None` usa coeficientes gerais e **não estima PWV**. Um valor informado seleciona as faixas de coeficientes; em sobreposições, calcula a média das estimativas de LST. Valores escalares inválidos geram erro; pixels inválidos de um raster viram NaN. |
| `mask` | Opcional; recomendado com dados de imagem | `None` | Adimensional (booleano) | **Máscara de exclusão de pixels**, na mesma grade: `True` rejeita o pixel, produzindo NaN na LST; `False` permite a avaliação, sujeita às outras validações. Neste exemplo vem de QA_PIXEL/QA_RADSAT e das verificações do leitor, detalhadas abaixo. Não passar a banda QA inteira como máscara booleana. |
| `unit` | Opcional | `"kelvin"` | K ou °C | `"kelvin"` ou `"celsius"`. Afeta apenas a saída; BT10/BT11 continuam entrando em K. A chamada acima escolhe Celsius explicitamente. |
| `return_diagnostics` | Opcional | `False` | — (booleano) | `False`: retorna somente o raster de LST. `True`: retorna dicionário com `lst`, `ndvi`, `emissivity_10`, `emissivity_11`, `invalid_mask`, `units` e `optical_contract`. `invalid_mask` é a rejeição final do cálculo, que pode incluir mais pixels que a máscara de entrada. |

## Alternativa — emissividades prontas

Fornecer e10/e11 específicos de cada banda, alinhados. Omitir emissivity_method e bandas ópticas; não combinar as duas formas de entrada.

```python
# e10/e11 são imagens já calculadas, alinhadas e específicas de cada banda.
lst = split_window(
    lst_method="du-2015",

    # Emissividades prontas — não fornecer B4/B5 ou emissivity_method
    emissivity_10=e10,
    emissivity_11=e11,

    # Temperaturas de brilho — K
    brightness_band_10=bt10,
    brightness_band_11=bt11,

    # Vapor d'água — g/cm²
    water_vapor=None,

    # Exclusão de pixels
    mask=mascara,

    # Saída
    unit="celsius",
    return_diagnostics=False,
)
```

### Detalhamento dos parâmetros da alternativa

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções para Du |
|---|---|---|---|---|
| `lst_method` | Obrigatório: selecionar o método | `None` (precisa informar) | — (seletor) | Usar `"du-2015"` neste notebook. |
| `emissivity_10` | Condicional: obrigatório junto com `emissivity_11` na entrada por emissividades prontas | `None` | Adimensional, (0,1] | Raster de emissividade da banda10. Para usá-lo, forneça também B11, deixe as bandas ópticas em `None` e omita `emissivity_method`. Não há preenchimento automático de uma banda a partir da outra. |
| `emissivity_11` | Condicional: obrigatório junto com `emissivity_10` na entrada por emissividades prontas | `None` | Adimensional, (0,1] | Raster de emissividade da banda11, com os parâmetros da própria banda. Deve ter a mesma forma e grade de BT10/BT11. |
| `brightness_band_10` | Obrigatório | Sem padrão | K (kelvin) | Raster de temperatura de brilho **TOA B10**, calibrado pelo MTL. Não fornecer DN, radiância, LST ou Celsius. |
| `brightness_band_11` | Obrigatório | Sem padrão | K (kelvin) | Raster de temperatura de brilho **TOA B11**, com a mesma forma e grade de B10. |
| `water_vapor` | Opcional | `None` | g/cm² | PWV escalar ou raster alinhado, entre **0 e 6,3**, inclusive. `None` usa coeficientes gerais e **não estima PWV**. Um valor informado seleciona as faixas de coeficientes; em sobreposições, calcula a média das estimativas de LST. Valores escalares inválidos geram erro; pixels inválidos de um raster viram NaN. |
| `mask` | Opcional; recomendado com dados de imagem | `None` | Adimensional (booleano) | **Máscara de exclusão de pixels**, na mesma grade: `True` rejeita o pixel, produzindo NaN na LST; `False` permite a avaliação, sujeita às outras validações. Neste exemplo vem de QA_PIXEL/QA_RADSAT e das verificações do leitor, detalhadas abaixo. Não passar a banda QA inteira como máscara booleana. |
| `unit` | Opcional | `"kelvin"` | K ou °C | `"kelvin"` ou `"celsius"`. Afeta apenas a saída; BT10/BT11 continuam entrando em K. A chamada acima escolhe Celsius explicitamente. |
| `return_diagnostics` | Opcional | `False` | — (booleano) | `False`: retorna somente o raster de LST. `True`: retorna dicionário com `lst`, `ndvi`, `emissivity_10`, `emissivity_11`, `invalid_mask`, `units` e `optical_contract`. `invalid_mask` é a rejeição final do cálculo, que pode incluir mais pixels que a máscara de entrada. |


In [ ]:
from pathlib import Path
import numpy as np
from pylstemp import split_window
from pylstemp.scene import LandsatScene

pasta = Path("CAMINHO_DA_CENA")
saida = Path("resultados/du_2015")
saida.mkdir(parents=True, exist_ok=True)

# Escolha da emissividade e suas entradas
metodo_emissividade = "gopinadh-2018"
grandeza_optica = "radiance"  # leitor: radiance ou reflectance (TOA)

# Vapor d'água — g/cm²
pwv = None                  # geral; ou escalar/raster [0,6.3] da cena

# Exclusão de pixels
excluir_agua = False         # True acrescenta a exclusão de água pela QA

# Saída
unidade = "celsius"          # kelvin ou celsius
diagnosticos = False         # True retorna também os intermediários

## Ler, calibrar e construir a máscara de qualidade

`LandsatScene` confere os arquivos e a grade e lê os coeficientes do MTL. `calibrated()` devolve as bandas ópticas calibradas, BT10/BT11 em K e uma máscara booleana de exclusão.

A variável `mascara` deste exemplo reúne:

- **QA_PIXEL:** fill/ausência de dados, nuvem dilatada, cirrus de alta confiança, nuvem, sombra de nuvem e neve, conforme os bits utilizados pela biblioteca.
- **QA_RADSAT:** saturação óptica em B2/B4/B5 e oclusão de terreno. Esses bits não são uma verificação de saturação térmica de B10/B11.
- **Ângulo solar inválido**, quando a banda SZA está presente, pelas verificações do leitor.
- **Água**, somente se `excluir_agua=True`. No padrão `False`, água não é excluída por esse bit; isso não garante adequação da parametrização de emissividade para água.

`True` significa **excluir**; `False` significa **não excluir pela máscara**. A máscara também pode incluir uma restrição adicional definida pelo usuário, como uma área externa ao recorte de interesse, desde que continue booleana e alinhada.

O cálculo ainda rejeita entradas numéricas inválidas, como BT não finita/não positiva ou emissividade fora de (0,1]. Ao passar arrays diretamente para `split_window`, `mask=None` não cria uma máscara de nuvens automaticamente. Pixels já convertidos para NaN durante a leitura/calibração continuam inválidos mesmo sem repassar a máscara.

In [ ]:
cena = LandsatScene(pasta)
opticas, bt, mascara = cena.calibrated(
    optical_quantity=grandeza_optica,
    exclude_water=excluir_agua,
)

bt10, bt11 = bt[10], bt[11]          # K
red, nir = opticas[4], opticas[5]    # mesma convenção óptica
print("Pixels excluídos pela máscara:", int(mascara.sum()))

## Aplicar Du com todas as opções visíveis

A chamada abaixo calcula emissividades a partir das bandas ópticas e mantém os mesmos grupos da função geral e da tabela. As opções da entrada por emissividades prontas são omitidas nessa forma de uso.

Os coeficientes foram elaborados para Landsat8; sua transferência para Landsat9 continua experimental. A seleção `"gopinadh-2018"` também exige justificar os limiares NDVI na cena e na convenção óptica adotada.

In [ ]:
resultado = split_window(
    lst_method="du-2015",

    # Cálculo da emissividade — requer vermelho/NIR
    emissivity_method=metodo_emissividade,
    band_4_red=red,
    band_5_nir=nir,
    optical_quantity=grandeza_optica,

    # Temperaturas de brilho — K
    brightness_band_10=bt10,
    brightness_band_11=bt11,

    # Vapor d'água — g/cm²
    water_vapor=pwv,

    # Exclusão de pixels
    mask=mascara,

    # Saída
    unit=unidade,
    return_diagnostics=diagnosticos,
)
imagem = resultado["lst"] if diagnosticos else resultado

## Salvar a imagem de temperatura da superfície

`write_geotiff` preserva a grade e grava a LST em float32, com NoData NaN. A etiqueta `units` recebe a mesma opção de `unit` utilizada no cálculo: `"kelvin"` ou `"celsius"`. Alterar somente a etiqueta do arquivo não converte os valores.

In [ ]:
cena.write_geotiff(saida / "du_2015.tif", imagem, units=unidade)
cena.close()
print("Pixels válidos na LST:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saída:", saida)

## Referência e limites

[Du et al. (2015) — artigo original](https://doi.org/10.3390/rs70100647).

Veja a [ficha metodológica](../docs/method-cards.md) para a equação, os coeficientes, o domínio e as adaptações. Execução numericamente correta não estabelece acurácia de campo. [Voltar ao índice](README.md).